# Data Wrangling #

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3
import re

# Connect to the database
conn = sqlite3.connect("./data/nutrition.db")
cur = conn.cursor()

In [2]:
food = pd.read_sql_query("SELECT * FROM food", conn)
food_nutrient = pd.read_sql_query("SELECT * FROM food_nutrient", conn)
nutrient = pd.read_sql_query("SELECT * FROM nutrient", conn)
wal_price = pd.read_sql_query("SELECT * FROM walmart_price", conn)
wf_price = pd.read_sql_query("SELECT * FROM wholefoods_price", conn)

In [3]:
def normalize_text(str):
    if pd.isna(str):
        return ""
    str = str.lower()
    str = re.sub(r'[^a-z0-9\s]', ' ', str)
    str = re.sub(r'\s+', ' ', str).strip()
    return str

food["clean_desc"] = food["description"].apply(normalize_text)
wal_price["clean_name"] = wal_price["product_name"].apply(normalize_text)
wf_price["clean_name"] = wf_price["product_name"].apply(normalize_text)

food["clean_brand_owner"] = food["brand_owner"].apply(normalize_text)
food["clean_brand"] = food["brand_name"].apply(normalize_text)
food["clean_subbrand"] = food["subbrand_name"].apply(normalize_text)

wal_price["clean_brand"] = wal_price["brand"].apply(normalize_text)
wf_price["clean_brand"] = wf_price["brand"].apply(normalize_text)

In [4]:
food_brand_owners = food["clean_brand_owner"].dropna().unique().tolist()
food_brands = food["clean_brand"].dropna().unique().tolist()
food_subbrands = food["clean_subbrand"].dropna().unique().tolist()

wal_brands = wal_price["clean_brand"].dropna().unique().tolist()
wf_brands = wf_price["clean_brand"].dropna().unique().tolist()

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

wf_brand_emb = model.encode(wf_brands, show_progress_bar=True)
wal_brand_emb = model.encode(wal_brands, show_progress_bar=True)

food_brand_owner_emb = model.encode(food_brand_owners, show_progress_bar=True)
food_brand_emb = model.encode(food_brands, show_progress_bar=True)
food_subbrand_emb = model.encode(food_subbrands, show_progress_bar=True)

/usr/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2174.63it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 238/238 [00:19<00:00, 12.23it/s]


In [13]:
from sentence_transformers import util
import numpy as np

wf_sim_matrix = util.cos_sim(wf_brand_emb, food_brand_emb)
wal_sim_matrix = util.cos_sim(wal_brand_emb, food_brand_emb)

wf_to_food = {}
for i, wf_brand in enumerate(wf_brands):
    best_idx = np.argmax(wf_sim_matrix[i]).item()
    best_match = food_brands[best_idx]
    best_score = wf_sim_matrix[i][best_idx].item()
    wf_to_food[wf_brand] = (best_match, best_score)

wal_to_food = {}
for i, wal_brand in enumerate(wal_brands):
    best_idx = np.argmax(wal_sim_matrix[i]).item()
    best_match = food_brands[best_idx]
    best_score = wal_sim_matrix[i][best_idx].item()
    wal_to_food[wal_brand] = (best_match, best_score)

wf_price['udsa_brand_match'] = wf_price['clean_brand'].map(lambda b: wf_to_food.get(b, (None, 0))[0])
wf_price['udsa_brand_score'] = wf_price['clean_brand'].map(lambda b: wf_to_food.get(b, (None, 0))[1])
wal_price['udsa_brand_match'] = wal_price['clean_brand'].map(lambda b: wal_to_food.get(b, (None, 0))[0])
wal_price['udsa_brand_score'] = wal_price['clean_brand'].map(lambda b: wal_to_food.get(b, (None, 0))[1])

In [ ]:
from rapidfuzz import process



In [ ]:
food_names = food["clean_desc"].tolist()

def get_candidates(query, k=5):
    get_candidates.calls += 1
    if get_candidates.calls < 10 or (get_candidates.calls < 100 and get_candidates.calls % 10 == 0) or (get_candidates.calls < 1000 and get_candidates.calls % 100 == 0) or (get_candidates.calls < 10000 and get_candidates.calls % 1000 == 0) or get_candidates.calls % 10000 == 0:
        print(get_candidates.calls)

    matches = process.extract(query, food_names, limit=k)
    return [(m[0], m[1]) for m in matches]

get_candidates.calls = 0

wf_price["candidates"] = wf_price["clean_name"].apply(get_candidates)

1
2
3
4
5
6
7
8
9
10


KeyboardInterrupt: 

In [ ]:
def compute_score(store_row, food_row, text_score):
    text_sim = text_score / 100

    brand_sim = 1 if store_row["clean_brand"] == food_row["clean_brand_owner"] or store_row["clean_brand"] == food_row["clean_brand"] or store_row["clean_brand"] == food_row["clean_subbrand"] else 0

    return 0.7 * text_sim + 0.3 * brand_sim

In [ ]:
def match_product(row, score_thresh):
    best_score = score_thresh
    best_fdc = None

    for cand_name, text_score in row["candidates"]:
        food_row = food[food["clean_desc"] == cand_name].iloc[0]

        score = compute_score(row, food_row, text_score)

        if score > best_score:
            best_score = score
            best_fdc = food_row["fdc_id"]

    return best_fdc, best_score

wal_price[["fdc_id", "match_score"]] = wal_price.apply(
    lambda row: pd.Series(match_product(row, 0)),
    axis=1
)

In [ ]:
conn.close()